# Notebook 01b — Control: diaObject catalogue vs saved light curve parquets

**Purpose**: Audit the consistency between:
1. The diaObject catalogues (`diaobj_catalogue.csv`, `diaobj_catalogue_gaia_stable.csv`) produced by notebook 01.
2. The light curve parquet files (`*_src.parquet`, `*_fp.parquet`) saved by notebook 01.
3. The Gaia-matched catalogue from notebook 08b (`fink_diaobj_gaia_join_matched.csv`).

**Key diagnostic questions**:
- For each diaObject in the catalogue, are there `src` and `fp` parquet rows?
- For each `src`/`fp` parquet row, is the corresponding diaObject in the catalogue?
- What is the NP_MIN filter impact? (objects present in catalogue but absent from parquets)
- Why does notebook 09b only find 11 Gaia-matched objects instead of the expected 146?

**Notebooks involved**: 01 (data production) → 08b (Gaia matching) → 09b (light curve plots)

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

print(f"pandas  version : {pd.__version__}")
print(f"numpy   version : {np.__version__}")

## 1. Configuration — paths


In [ ]:
# ── Input directories ────────────────────────────────────────────────────────
DIR_DATA_01 = "data_FINK_BLOCK_LC_01"  # produced by notebook 01
DIR_DATA_08b = "data_FINK_BLOCK_LC_08b"  # produced by notebook 08b

# ── Key files ────────────────────────────────────────────────────────────────
# Catalogues
FILE_CAT_FULL = os.path.join(DIR_DATA_01, "diaobj_catalogue.csv")
FILE_CAT_GAIA_STABLE = os.path.join(DIR_DATA_01, "diaobj_catalogue_gaia_stable.csv")
FILE_GAIA_MATCHED = os.path.join(DIR_DATA_08b, "fink_diaobj_gaia_join_matched.csv")

# All expected parquet file patterns in DIR_DATA_01
# (one file per Fink group × {src, fp})
PARQUET_SUFFIXES = ["_src.parquet", "_fp.parquet"]

# Fink column names used in parquet files
COL_OBJ = "r:diaObjectId"
COL_MJD = "r:midpointMjdTai"
COL_BAND = "r:band"

print("Paths configured.")
for f in [FILE_CAT_FULL, FILE_CAT_GAIA_STABLE, FILE_GAIA_MATCHED]:
    exists = os.path.exists(f)
    print(f"  {'OK' if exists else 'MISSING':7s} {f}")

## 2. Load catalogues


In [ ]:
# ── Full diaObject catalogue (all groups) ────────────────────────────────────
df_cat_full = pd.read_csv(FILE_CAT_FULL)
df_cat_full["diaObjectId"] = df_cat_full["diaObjectId"].astype(str)
print(f"Full catalogue          : {len(df_cat_full):,} objects")
print(f"  Groups:")
print(df_cat_full["group"].value_counts().to_string())
print()

# ── Gaia-stable sub-catalogue (saved by notebook 01) ─────────────────────────
df_cat_gaia = pd.read_csv(FILE_CAT_GAIA_STABLE)
df_cat_gaia["diaObjectId"] = df_cat_gaia["diaObjectId"].astype(str)
print(f"Gaia-stable sub-cat     : {len(df_cat_gaia):,} objects")
print(f"  Groups:")
print(df_cat_gaia["group"].value_counts().to_string())
print()

# ── Gaia-matched catalogue from notebook 08b ─────────────────────────────────
df_gaia_matched = pd.read_csv(FILE_GAIA_MATCHED)
df_gaia_matched["diaObjectId"] = df_gaia_matched["diaObjectId"].astype(str)
print(f"Gaia-matched (08b)      : {len(df_gaia_matched):,} objects")
print(f"  Groups:")
print(df_gaia_matched["group"].value_counts().to_string())

## 3. Inventory of parquet files saved by notebook 01


In [ ]:
# ── List all parquet files in DIR_DATA_01 ────────────────────────────────────
parquet_files = sorted(f for f in os.listdir(DIR_DATA_01) if f.endswith(".parquet"))

print(f"Parquet files found in {DIR_DATA_01}/ ({len(parquet_files)} total):")
for f in parquet_files:
    size_kb = os.path.getsize(os.path.join(DIR_DATA_01, f)) / 1024
    print(f"  {f:<60s} {size_kb:8.1f} kB")

## 4. Load all parquets and build diaObjectId × file inventory


In [ ]:
# ── Load all parquets and record which diaObjectIds appear in each file ───────
parquet_inventory = {}  # fname → {'group': str, 'kind': str, 'oids': set, 'n_rows': int}

for fname in parquet_files:
    path = os.path.join(DIR_DATA_01, fname)
    try:
        df_tmp = pd.read_parquet(path, columns=[COL_OBJ])
        oids = set(df_tmp[COL_OBJ].astype(str).unique())
        n_rows = len(df_tmp)
    except Exception as e:
        print(f"  ERROR reading {fname}: {e}")
        oids = set()
        n_rows = 0

    # Derive group name and kind (src / fp) from filename
    kind = "src" if fname.endswith("_src.parquet") else "fp"
    group = fname.replace("_src.parquet", "").replace("_fp.parquet", "")

    parquet_inventory[fname] = {
        "group": group,
        "kind": kind,
        "oids": oids,
        "n_rows": n_rows,
    }
    print(f"  {fname:<60s} {n_rows:6,} rows  {len(oids):4,} unique diaObjectIds")

# Build union of all diaObjectIds that appear in any parquet
all_oids_in_parquets = set()
src_oids_in_parquets = set()
fp_oids_in_parquets = set()

for info in parquet_inventory.values():
    all_oids_in_parquets |= info["oids"]
    if info["kind"] == "src":
        src_oids_in_parquets |= info["oids"]
    else:
        fp_oids_in_parquets |= info["oids"]

print()
print(f"Total unique diaObjectIds across ALL parquets : {len(all_oids_in_parquets):,}")
print(f"  in src parquets                            : {len(src_oids_in_parquets):,}")
print(f"  in fp  parquets                            : {len(fp_oids_in_parquets):,}")

## 5. Cross-check: catalogue ↔ parquets


In [ ]:
# ── Set of diaObjectIds per catalogue ────────────────────────────────────────
cat_full_oids = set(df_cat_full["diaObjectId"].astype(str))
cat_gaia_oids = set(df_cat_gaia["diaObjectId"].astype(str))
cat_gaia_matched_oids = set(df_gaia_matched["diaObjectId"].astype(str))

print("=" * 65)
print("CROSS-CHECK SUMMARY")
print("=" * 65)

# --- 5.1 Full catalogue vs parquets ---
in_cat_not_in_parquets = cat_full_oids - all_oids_in_parquets
in_parquets_not_in_cat = all_oids_in_parquets - cat_full_oids

print(f"\n--- Full catalogue vs ALL parquets ---")
print(f"  Objects in catalogue                    : {len(cat_full_oids):,}")
print(f"  Objects in any parquet                  : {len(all_oids_in_parquets):,}")
print(
    f"  In catalogue BUT NOT in any parquet     : {len(in_cat_not_in_parquets):,}  <- NP_MIN filter candidates"
)
print(f"  In parquets BUT NOT in catalogue        : {len(in_parquets_not_in_cat):,}  <- unexpected")

# --- 5.2 Gaia-stable sub-catalogue vs parquets ---
in_gaia_cat_not_in_src = cat_gaia_oids - src_oids_in_parquets
in_gaia_cat_not_in_fp = cat_gaia_oids - fp_oids_in_parquets
in_gaia_cat_in_src = cat_gaia_oids & src_oids_in_parquets
in_gaia_cat_in_fp = cat_gaia_oids & fp_oids_in_parquets

print(f"\n--- Gaia-stable sub-catalogue vs src/fp parquets ---")
print(f"  Objects in Gaia-stable catalogue        : {len(cat_gaia_oids):,}")
print(f"  Objects with src parquet rows           : {len(in_gaia_cat_in_src):,}")
print(f"  Objects with fp  parquet rows           : {len(in_gaia_cat_in_fp):,}")
print(f"  In Gaia-stable cat BUT missing src      : {len(in_gaia_cat_not_in_src):,}")
print(f"  In Gaia-stable cat BUT missing fp       : {len(in_gaia_cat_not_in_fp):,}")

# --- 5.3 Gaia-MATCHED (08b) vs src parquets ---
matched_in_src = cat_gaia_matched_oids & src_oids_in_parquets
matched_not_in_src = cat_gaia_matched_oids - src_oids_in_parquets

print(f"\n--- Gaia-MATCHED (08b, {len(cat_gaia_matched_oids)} objects) vs src parquets ---")
print(f"  Matched objects found in src parquets   : {len(matched_in_src):,}  <- expected 146")
print(f"  Matched objects NOT in src parquets     : {len(matched_not_in_src):,}  <- THE MISSING ONES")

## 6. Per-group breakdown of catalogue vs parquet coverage


In [ ]:
# ── Build per-group summary ──────────────────────────────────────────────────
rows = []
for grp, sub in df_cat_full.groupby("group"):
    oids_grp = set(sub["diaObjectId"].astype(str))

    # Look for the expected parquet files for this group
    safe_grp = grp.replace("/", "_")
    src_fname = f"{safe_grp}_src.parquet"
    fp_fname = f"{safe_grp}_fp.parquet"

    src_info = parquet_inventory.get(src_fname, {})
    fp_info = parquet_inventory.get(fp_fname, {})

    src_oids = src_info.get("oids", set())
    fp_oids = fp_info.get("oids", set())

    rows.append(
        {
            "group": grp,
            "n_in_catalogue": len(oids_grp),
            "n_in_src": len(oids_grp & src_oids),
            "n_in_fp": len(oids_grp & fp_oids),
            "n_missing_src": len(oids_grp - src_oids),
            "n_missing_fp": len(oids_grp - fp_oids),
            "src_file_exists": os.path.exists(os.path.join(DIR_DATA_01, src_fname)),
            "fp_file_exists": os.path.exists(os.path.join(DIR_DATA_01, fp_fname)),
        }
    )

df_grp_summary = pd.DataFrame(rows).sort_values("n_missing_src", ascending=False)
print("Per-group catalogue vs parquet coverage:")
print(df_grp_summary.to_string(index=False))

## 7. NP_MIN filter diagnosis — nDiaSources for objects missing from parquets

Notebook 01 applies `NP_MIN = 100` when fetching light curves via the Fink API.
Objects in the catalogue that are absent from the parquets likely failed this filter.


In [ ]:
# Objects that are in the catalogue but have NO src parquet rows
missing_from_src = df_cat_full[df_cat_full["diaObjectId"].astype(str).isin(in_cat_not_in_parquets)].copy()

print(f"Objects in catalogue but absent from ALL parquets: {len(missing_from_src)}")

if "nDiaSources" in missing_from_src.columns:
    print(f"\nnDiaSources statistics for missing objects:")
    print(missing_from_src["nDiaSources"].describe().to_string())
    print(f"\nBreakdown by group:")
    print(missing_from_src.groupby("group")["nDiaSources"].agg(["count", "min", "max", "median"]).to_string())
else:
    print("WARNING: nDiaSources column not found in catalogue.")

# And specifically: which Gaia-matched objects from 08b are missing from src?
missing_matched = df_gaia_matched[df_gaia_matched["diaObjectId"].astype(str).isin(matched_not_in_src)].copy()

print(f"\nGaia-matched objects (08b) absent from src parquets: {len(missing_matched)}")
if "nDiaSources" in missing_matched.columns:
    print("nDiaSources for missing Gaia-matched objects:")
    print(missing_matched["nDiaSources"].describe().to_string())
    print("\nPer group:")
    print(missing_matched.groupby("group")["nDiaSources"].agg(["count", "min", "max", "median"]).to_string())

## 8. Type-mismatch diagnosis — notebook 09b intersection bug

Notebook 09b builds `gaia_matched_ids` as a set of **strings** (`.astype(str)`),
then compares against `df[COL_OBJ]` where `COL_OBJ = 'r:diaObjectId'`.
If the parquet stores this column as `int64` (or `object` with different formatting),
the `.isin()` comparison can silently fail.


In [ ]:
# ── Inspect actual dtype of COL_OBJ in src parquets ─────────────────────────
print("dtype of r:diaObjectId in each src parquet:\n")

src_parquet_files = [f for f in parquet_files if f.endswith("_src.parquet")]

for fname in src_parquet_files:
    path = os.path.join(DIR_DATA_01, fname)
    df_tmp = pd.read_parquet(path, columns=[COL_OBJ])
    print(f"  {fname:<55s}  dtype={df_tmp[COL_OBJ].dtype}")
    sample = df_tmp[COL_OBJ].dropna().head(3).tolist()
    print(f"    sample values: {sample}")

print()

# ── Inspect dtype in the 08b matched CSV ────────────────────────────────────
print(f"dtype of diaObjectId in {FILE_GAIA_MATCHED}:")
print(f"  {df_gaia_matched['diaObjectId'].dtype}")
sample_matched = df_gaia_matched["diaObjectId"].head(3).tolist()
print(f"  sample values: {sample_matched}")

# ── Simulate the 09b comparison logic ────────────────────────────────────────
print("\nSimulating notebook 09b comparison logic...")

# Reconstruct gaia_matched_ids exactly as notebook 09b does
gaia_matched_ids_str = set(df_gaia_matched["diaObjectId"].astype(str).values)

# Load a src parquet as 09b would
sample_src_file = src_parquet_files[0] if src_parquet_files else None
if sample_src_file:
    df_sample = pd.read_parquet(os.path.join(DIR_DATA_01, sample_src_file), columns=[COL_OBJ])

    # Method 1: no type cast (potential bug — isin() with mixed int64 vs str set)
    n_match_raw = df_sample[COL_OBJ].isin(gaia_matched_ids_str).sum()

    # Method 2: cast src column to str first (the correct approach)
    n_match_str = df_sample[COL_OBJ].astype(str).isin(gaia_matched_ids_str).sum()

    print(f"  File: {sample_src_file}")
    print(f"  isin() without str cast : {n_match_raw} matches  <- may be 0 if dtype mismatch")
    print(f"  isin() WITH    str cast : {n_match_str} matches  <- correct")

    unique_raw = df_sample[COL_OBJ].nunique()
    unique_str = df_sample[COL_OBJ].astype(str).nunique()
    print(f"  Unique diaObjectIds (raw dtype)  : {unique_raw}")
    print(f"  Unique diaObjectIds (as str)     : {unique_str}")

## 9. Group coverage — which groups does notebook 09b actually load?

Notebook 09b only loads **two** parquet files:
- `gaia_star_stable_hq_src.parquet`
- `gaia_nophotgstar_stable_unknown_parallax_src.parquet`

Other Gaia groups (`gaia_brightstar_stable_unknown_parallax`, `gaia_star_variable`, etc.)
are **never loaded**, even if their parquets exist and contain Gaia-matched objects.


In [ ]:
# Groups that 09b loads vs groups that exist in parquets
GROUPS_LOADED_BY_09b = {
    "gaia_star_stable_hq",
    "gaia_nophotgstar_stable_unknown_parallax",
}

# All groups present in src parquets
groups_in_src = set(
    info["group"] for info in parquet_inventory.values() if info["kind"] == "src" and len(info["oids"]) > 0
)

print("Groups present in src parquets:")
for g in sorted(groups_in_src):
    loaded = "  OK loaded by 09b" if g in GROUPS_LOADED_BY_09b else "  NOT loaded by 09b"
    n_oids = len(parquet_inventory.get(g + "_src.parquet", {}).get("oids", set()))
    print(f"  {g:<55s} {n_oids:4,} objects   {loaded}")

# How many Gaia-matched objects are in groups NOT loaded by 09b?
print()
missing_groups = groups_in_src - GROUPS_LOADED_BY_09b
n_matched_in_missing_groups = df_gaia_matched[df_gaia_matched["group"].isin(missing_groups)].shape[0]
print(f"Gaia-matched objects in groups NOT loaded by 09b: {n_matched_in_missing_groups}")
if n_matched_in_missing_groups > 0:
    print(df_gaia_matched[df_gaia_matched["group"].isin(missing_groups)]["group"].value_counts().to_string())

## 10. Full reconciliation: one row per Gaia-matched diaObject

For each of the 146 Gaia-matched objects (08b), check:
- Is it in the src parquets? In which file/group?
- Is it in the fp parquets?
- Was it loaded by notebook 09b?
- If not, why? (missing from parquet / wrong group / type mismatch)


In [ ]:
# Build a comprehensive status table for each Gaia-matched object
status_rows = []

for _, obj_row in df_gaia_matched.iterrows():
    oid_str = str(obj_row["diaObjectId"])
    grp = str(obj_row.get("group", "?"))

    in_src = oid_str in src_oids_in_parquets
    in_fp = oid_str in fp_oids_in_parquets

    # Which specific src file contains this object?
    src_files_containing = [
        fname
        for fname, info in parquet_inventory.items()
        if info["kind"] == "src" and oid_str in info["oids"]
    ]

    loaded_by_09b = grp in GROUPS_LOADED_BY_09b and in_src

    # Root cause for objects not found by 09b
    if loaded_by_09b:
        root_cause = "OK"
    elif not in_src:
        ndias = obj_row.get("nDiaSources", None)
        if ndias is not None and pd.notna(ndias) and float(ndias) < 100:
            root_cause = f"NP_MIN filter (nDiaSrc={int(ndias)}<100)"
        else:
            root_cause = f"Missing from src parquets (nDiaSrc={ndias})"
    else:
        root_cause = f'Group "{grp}" not loaded by 09b'

    status_rows.append(
        {
            "diaObjectId": oid_str,
            "group": grp,
            "nDiaSources": obj_row.get("nDiaSources", np.nan),
            "gaia_G_mag": obj_row.get("gaia_phot_g_mean_mag", np.nan),
            "in_src_parquet": in_src,
            "in_fp_parquet": in_fp,
            "loaded_by_09b": loaded_by_09b,
            "src_files": ", ".join(src_files_containing) if src_files_containing else "NONE",
            "root_cause": root_cause,
        }
    )

df_status = pd.DataFrame(status_rows)

print("Reconciliation table (146 Gaia-matched objects):")
print(f"  Loaded correctly by 09b          : {df_status['loaded_by_09b'].sum()}")
print(f"  NOT loaded by 09b                : {(~df_status['loaded_by_09b']).sum()}")
print()
print("Root causes breakdown:")
print(df_status["root_cause"].value_counts().to_string())

In [ ]:
# Display the full table of objects not found by 09b
df_not_found = df_status[~df_status["loaded_by_09b"]].sort_values(["root_cause", "nDiaSources"])
print(f"Objects NOT loaded by notebook 09b ({len(df_not_found)}):")
print(
    df_not_found[
        [
            "diaObjectId",
            "group",
            "nDiaSources",
            "gaia_G_mag",
            "in_src_parquet",
            "in_fp_parquet",
            "src_files",
            "root_cause",
        ]
    ].to_string(index=False)
)

## 11. Summary visualisation


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# ── Panel 1: Root-cause pie chart ────────────────────────────────────────────
ax = axes[0]
cause_counts = df_status["root_cause"].value_counts()


def shorten(label):
    if label == "OK":
        return "Found by 09b"
    if "NP_MIN" in label:
        return "NP_MIN filter (<100 detections)"
    if "not loaded by 09b" in label:
        return "Wrong group (not loaded by 09b)"
    return label


cause_counts.index = [shorten(x) for x in cause_counts.index]
cause_counts = cause_counts.groupby(level=0).sum()
colors = [
    "#2ecc71" if "Found" in x else "#e74c3c" if "NP_MIN" in x else "#e67e22" for x in cause_counts.index
]
ax.pie(
    cause_counts.values,
    labels=cause_counts.index,
    colors=colors,
    autopct="%1.0f%%",
    startangle=140,
    textprops={"fontsize": 9},
)
ax.set_title(f"Why 09b finds fewer than 146 objects\n(total = {len(df_status)})", fontsize=10)

# ── Panel 2: nDiaSources distribution — found vs missing ─────────────────────
ax = axes[1]
bins = np.linspace(0, df_status["nDiaSources"].dropna().quantile(0.99) + 1, 30)

found = df_status[df_status["loaded_by_09b"]]["nDiaSources"].dropna()
missing = df_status[~df_status["loaded_by_09b"]]["nDiaSources"].dropna()

ax.hist(
    found,
    bins=bins,
    histtype="stepfilled",
    alpha=0.6,
    color="#2ecc71",
    label=f"Found by 09b (N={len(found)})",
)
ax.hist(
    missing,
    bins=bins,
    histtype="stepfilled",
    alpha=0.6,
    color="#e74c3c",
    label=f"Missing from 09b (N={len(missing)})",
)
ax.axvline(100, ls="--", color="k", lw=1.2, label="NP_MIN = 100")
ax.set_xlabel("nDiaSources")
ax.set_ylabel("N Gaia-matched objects")
ax.set_title("nDiaSources distribution — found vs missing")
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig("01b_reconciliation_summary.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved: 01b_reconciliation_summary.png")

## 12. Proposed fixes for notebook 09b

Based on the audit above, three independent issues may reduce the 146 → 11 count.
Apply the relevant fix(es) in notebook 09b.

### Fix A — Type-safe set membership (always apply)
```python
# In notebook 09b, cell 16: always cast COL_OBJ to str before .isin()
df['_oid_str'] = df[COL_OBJ].astype(str)
df_gaia = df[df['_oid_str'].isin(gaia_matched_ids)].copy()   # gaia_matched_ids already str
```
(Cell 16 of 09b already has this — verify it is not commented out.)

### Fix B — Load ALL Gaia-group parquets, not just hq + unknownparallax
```python
import glob

# Auto-discover all Gaia parquets instead of hardcoding two filenames
gaia_src_files = sorted(glob.glob(os.path.join(DIR_DATA_FLUX, 'gaia_*_src.parquet')))
frames = []
for path in gaia_src_files:
    tag = os.path.basename(path).replace('_src.parquet', '')
    df_tmp = pd.read_parquet(path)
    df_tmp['gaia_origin'] = tag
    frames.append(df_tmp)
    print(f'  Loaded {tag}: {len(df_tmp):,} rows')
df_src = pd.concat(frames, ignore_index=True)
```

### Fix C — Re-run notebook 01 with NP_MIN lowered for Gaia objects
If many Gaia-matched objects have `nDiaSources < 100`, add a dedicated
secondary fetch in notebook 01 using a lower threshold:
```python
NP_MIN_GAIA = 5   # lower threshold for Gaia-priority calibration objects
```
Or fetch Gaia-matched objects explicitly by `diaObjectId` via `/api/v1/sources`.


In [ ]:
# Save the full reconciliation table for inspection in other notebooks
out_path = os.path.join(DIR_DATA_01, "01b_gaia_matched_reconciliation.csv")
df_status.to_csv(out_path, index=False)
print(f"Saved reconciliation table -> {out_path}  ({len(df_status)} rows)")
print(
    df_status[["diaObjectId", "group", "in_src_parquet", "loaded_by_09b", "root_cause"]]
    .head(30)
    .to_string(index=False)
)